# Transform Races Data
1. Read bronze `races` table
2. Keep only the columns required for analytics (Drop `url` column)
3. Standardize column names using snake_case
4. Rename columns to make them more meaningful 
5. Remove duplicate records
6. Transform values of column `race_name` to Title Case
7. Write the transformed data to silver `race` table

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.races'
silver_table = f'{catalog_name}.{silver_schema}.races'

### Step 1 - Read bronze `races` table

In [0]:
races_df = spark.table(bronze_table)

### Step 2 - Keep only the columns required for analytics (Drop `url` column)

In [0]:
from pyspark.sql import functions as F
races_selected_df = races_df.drop(
    F.col("url")
)

In [0]:
races_selected_df = races_df.select(
    F.col('season'),
    F.col('round'),
    F.col('raceName'),
    F.col('date'),
    F.col('circuitId'),
    F.col('ingestion_timestamp'),
    F.col('source_file')
)

### Step 3 & 4 - Standardize column names

In [0]:
races_renamed_df = (
    races_selected_df
        .withColumnsRenamed({
            'raceName' : 'race_name',
            'circuitId': 'circuit_id',
            'date': 'race_date'
        })
)

### Step 5 - Remove duplicate records

In [0]:
races_distinct_df = races_renamed_df.dropDuplicates(['season', 'round'])

In [0]:
display(races_distinct_df)

In [0]:
races_distinct_df = races_distinct_df.filter(
    F.col('circuit_id').isNotNull()
)
display(races_distinct_df)

### Step 6 - Transform values of column `race_name` to Title Case

In [0]:
races_final_df = (
    races_distinct_df
    .withColumn('race_name', F.initcap(F.col("race_name")))
    )

In [0]:
display(races_final_df)

# Step 8 - Write the transformed data to silver `races` table

In [0]:
(
    races_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))